In [1]:
from __future__ import annotations

import math
import os
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy, rowcol
from pyproj import Transformer

from geotessera import GeoTessera
import geopandas as gpd
import time
from scipy.ndimage import distance_transform_edt
from skimage.measure import block_reduce

import os
os.environ['AWS_PROFILE'] = 'ctrees'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'YES'
os.environ['VSI_CURL_ALLOW_CREDENTIALS'] = 'YES'

import boto3
s3 = boto3.client('s3')


In [2]:
# Get training tiles
training_tiles_path = 's3://ctrees-development/activity/tessera/planet_nicfi_logging_tessera_pantropical_v1_testing.gpkg'
training_tiles = gpd.read_file(training_tiles_path)
tiles_id = training_tiles['name'].tolist()
print(f"There are {len(tiles_id)} training tiles.")

There are 10 training tiles.


In [3]:
# Get corresponding planet tiles
planet_bucket ='ctrees-activity-product'
planet_prefix = 'logging/v1_2024_update/cumulative/2015-12-01-2025-01-01/'
planet_file_dict = {tile_id: f's3://{planet_bucket}/{planet_prefix}logging_cumulative_PAGE_{tile_id}_START_2015-12-01_END_2025-01-01.tif' for tile_id in tiles_id}
print(f"There are {len(planet_file_dict)} planet files corresponding to the training tiles.")


There are 10 planet files corresponding to the training tiles.


In [ ]:

# Label preprocessing

def remap_labels(arr: np.ndarray) -> np.ndarray:
    """
    Map Planet labels:
      240, 245 -> 1
      0        -> 0
      others   -> 255 (nodata)
    """
    t0 = time.time()
    out = np.full(arr.shape, 255, dtype=np.uint8)
    out[arr == 0] = 0
    out[(arr == 240) | (arr == 245)] = 1
    print(f"Remapped labels in {time.time() - t0:.2f} seconds")
    return out


def aggregate_to_coarser_resolution(arr, scale_factor):
    """"Aggregate the input array to a coarser resolution by averaging over non-overlapping blocks.
    Parameters:
        arr (np.ndarray): Input 2D array to be aggregated.
        scale_factor (int): The factor by which to downsample the array. 
                            For example, a scale_factor of 2 will reduce the resolution by half.    
    Returns:
        np.ndarray: The aggregated 2D array at the coarser resolution.
    """

    # downsample by averaging over 2x2 blocks
    coarser_array = block_reduce(arr, block_size=(scale_factor, scale_factor), func=np.mean)
    print(f'uniqure values after aggregation: {np.unique(coarser_array)}')

    # get only the pixels that are equasl to 1 
    coarser_array = np.where(coarser_array > 1, 255, coarser_array).astype(np.uint8)
    print(f'uniqure values after final thresholding: {np.unique(coarser_array)}')

    return coarser_array

def aggregated_transform_2x2(src_transform):
    """
    Update affine transform after 2x2 aggregation.
    """
    print(f"Updating affine transform for 2x2 aggregation...")
    return src_transform * src_transform.scale(2, 2)

def buffer_by_proximity(binary_array, buffer_size):
    """
    Buffer binary mask by proximity using distance transform.
    For each pixel, compute distance to nearest non-zero pixel.
    If distance <= buffer_size, set to 0 (mask out).
    Else set to 1 (keep).
    """
    if binary_array is not None:
        proximity = distance_transform_edt(binary_array)
        proximity[np.where(proximity <= buffer_size)] = 0 
        proximity[np.where(proximity != 0)] = 1 
        proximity = np.where(proximity != 0,1,0).astype(np.uint8)
    else:
        proximity = None
    return proximity



# Sample balanced points

def sample_balanced_pixels(
    label_agg: np.ndarray,
    transform,
    crs,
    frac1: float = 0.5,
    frac0: float = 0.5,
    max_samples_per_class: Optional[int] = 50000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Randomly sample a fraction of class-1 and class-0 pixels.

    Returns a table with:
      class, row, col, x_proj, y_proj, lon, lat
    """
    t0 = time.time()
    print(f"Sampling balanced pixels with frac1={frac1}, frac0={frac0}, max_samples_per_class={max_samples_per_class}...")
    rng = np.random.default_rng(seed)

    idx1 = np.argwhere(label_agg == 1)
    idx0 = np.argwhere(label_agg == 0)

    n1 = int(len(idx1) * frac1)
    n0 = int(len(idx0) * frac0)

    if max_samples_per_class is not None:
        n1 = min(n1, max_samples_per_class)
        n0 = min(n0, max_samples_per_class)

    s1 = idx1[rng.choice(len(idx1), size=n1, replace=False)]
    s0 = idx0[rng.choice(len(idx0), size=n0, replace=False)]

    rows = np.concatenate([s1[:, 0], s0[:, 0]])
    cols = np.concatenate([s1[:, 1], s0[:, 1]])
    labels = np.concatenate([
        np.ones(len(s1), dtype=np.uint8),
        np.zeros(len(s0), dtype=np.uint8),
    ])

    xs, ys = xy(transform, rows, cols, offset="center")
    xs = np.asarray(xs)
    ys = np.asarray(ys)

    transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    lons, lats = transformer.transform(xs, ys)

    df = pd.DataFrame({
        "class": labels,
        "row": rows.astype(np.int32),
        "col": cols.astype(np.int32),
        "x_proj": xs.astype(np.float64),
        "y_proj": ys.astype(np.float64),
        "lon": np.asarray(lons, dtype=np.float64),
        "lat": np.asarray(lats, dtype=np.float64),
        "planet_crs": str(crs),
    })

    print(f"Sampled balanced pixels in {time.time() - t0:.2f} seconds")
    return df



# Tessera tile grouping

def tessera_tile_center_from_lonlat(
    lon: np.ndarray, lat: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Get Tessera tile center from lon/lat.
    Tessera tile grid is 0.1° x 0.1° with centers at x.05 / y.05.

    Example:
      lon=-77.52 -> tile center -77.55
      lat=-4.33  -> tile center -4.35
    """
    t0 = time.time()
    print(f"Computing Tessera tile centers from lon/lat...")
    eps = 1e-12
    tile_lon = np.floor((lon - eps) * 10.0) / 10.0 + 0.05
    tile_lat = np.floor((lat - eps) * 10.0) / 10.0 + 0.05
    print(f"Computed Tessera tile centers in {time.time() - t0:.2f} seconds")
    return np.round(tile_lon, 2), np.round(tile_lat, 2)


def add_tessera_tile_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add Tessera tile center columns to the point table.
    """
    t0 = time.time()
    print(f"Adding Tessera tile center columns to the dataframe...")
    tile_lon, tile_lat = tessera_tile_center_from_lonlat(
        df["lon"].to_numpy(),
        df["lat"].to_numpy(),
    )
    out = df.copy()
    out["tessera_tile_lon"] = tile_lon
    out["tessera_tile_lat"] = tile_lat
    print(f"Added Tessera tile center columns in {time.time() - t0:.2f} seconds")
    return out



# Optimized tile-grouped embedding extraction

def fetch_embeddings_grouped_by_tile(
    df_points: pd.DataFrame,
    year: int,
    cache_dir: Optional[str] = None,
) -> pd.DataFrame:
    """
    Fetch each Tessera tile once, then extract all grouped points.
    
    Optimized extraction:
      - group sample points by Tessera tile center
      - fetch each embedding tile once
      - transform all points in the group to the tile CRS
      - extract embeddings with vectorized indexing

    Returns:
      original dataframe + emb_000...emb_127 + tessera metadata
    """
    t0 = time.time()
    print(f"Fetching Tessera embeddings grouped by tile for year={year}...")
    gt = GeoTessera(cache_dir=cache_dir) if cache_dir else GeoTessera()

    min_lon = float(df_points["lon"].min())
    min_lat = float(df_points["lat"].min())
    max_lon = float(df_points["lon"].max())
    max_lat = float(df_points["lat"].max())
    bbox = (min_lon, min_lat, max_lon, max_lat)

    tiles_to_fetch = gt.registry.load_blocks_for_region(bounds=bbox, year=year)

    # Build tile dictionary
    # key: (tile_lon, tile_lat)
    tile_dict: Dict[Tuple[float, float], Dict[str, object]] = {}
    for y, tile_lon, tile_lat, embedding, crs, transform in gt.fetch_embeddings(tiles_to_fetch):
        key = (round(float(tile_lon), 2), round(float(tile_lat), 2))
        tile_dict[key] = {
            "year": y,
            "tile_lon": round(float(tile_lon), 2),
            "tile_lat": round(float(tile_lat), 2),
            "embedding": embedding,
            "crs": crs,
            "transform": transform,
        }

    if not tile_dict:
        raise ValueError("No Tessera tiles fetched for the sample bbox.")

    # Prepare output arrays
    n = len(df_points)
    emb_out = np.full((n, 128), np.nan, dtype=np.float32)

    tessera_x = np.full(n, np.nan, dtype=np.float64)
    tessera_y = np.full(n, np.nan, dtype=np.float64)
    tessera_row = np.full(n, -1, dtype=np.int32)
    tessera_col = np.full(n, -1, dtype=np.int32)
    tessera_crs = np.empty(n, dtype=object)

    # Group by tile
    for (tile_lon, tile_lat), group in df_points.groupby(
        ["tessera_tile_lon", "tessera_tile_lat"], sort=False
    ):
        key = (round(float(tile_lon), 2), round(float(tile_lat), 2))
        if key not in tile_dict:
            continue

        tile = tile_dict[key]
        embedding = tile["embedding"]
        tile_crs = tile["crs"]
        tile_transform = tile["transform"]

        h, w, nb = embedding.shape
        if nb != 128:
            raise ValueError(f"Expected 128 embedding bands, got {nb}")

        idx = group.index.to_numpy()
        lons = group["lon"].to_numpy()
        lats = group["lat"].to_numpy()

        # Transform point lon/lat -> tile native CRS
        transformer = Transformer.from_crs("EPSG:4326", tile_crs, always_xy=True)
        xs, ys = transformer.transform(lons, lats)
        xs = np.asarray(xs)
        ys = np.asarray(ys)

        # Convert coordinates to tile pixel indices
        rows, cols = rowcol(tile_transform, xs, ys)
        rows = np.asarray(rows)
        cols = np.asarray(cols)

        valid = (
            (rows >= 0) & (rows < h) &
            (cols >= 0) & (cols < w)
        )

        if np.any(valid):
            valid_idx = idx[valid]
            valid_rows = rows[valid]
            valid_cols = cols[valid]

            emb_out[valid_idx, :] = embedding[valid_rows, valid_cols, :]
            tessera_x[valid_idx] = xs[valid]
            tessera_y[valid_idx] = ys[valid]
            tessera_row[valid_idx] = valid_rows
            tessera_col[valid_idx] = valid_cols
            tessera_crs[valid_idx] = str(tile_crs)

    # Build final table
    emb_cols = [f"emb_{i:03d}" for i in range(128)]
    emb_df = pd.DataFrame(emb_out, columns=emb_cols, index=df_points.index)

    out = pd.concat([df_points.reset_index(drop=True), emb_df.reset_index(drop=True)], axis=1)
    out["tessera_x"] = tessera_x
    out["tessera_y"] = tessera_y
    out["tessera_row"] = tessera_row
    out["tessera_col"] = tessera_col
    out["tessera_crs"] = tessera_crs

    print(f"Fetched Tessera embeddings and merged with points in {time.time() - t0:.2f} seconds")
    return out



# Per-raster processing

def process_one_planet_raster(
    planet_raster: str,
    tessera_year: int,
    frac1: float = 0.5,
    frac0: float = 0.5,
    max_samples_per_class: Optional[int] = 10000,
    seed: int = 42,
    cache_dir: Optional[str] = None,
) -> pd.DataFrame:
    """
    Process one Planet raster into one training dataframe.

    Full optimized pipeline:

      1) Open projected Planet raster (~5 m pixels)
      2) Remap labels:
           240,245 -> 1
           0       -> 0
           others  -> 255
      3) Exact 2x2 block aggregation (~9.55 m from 4.777m source)
      4) Randomly sample balanced class 1 / class 0 pixels
      5) Convert sampled pixel centers to lon/lat
      6) Compute Tessera tile for each point
      7) Fetch each Tessera tile once and extract all grouped points
      8) Return final table for LightGBM
    """
    t0 = time.time()
    print(f"Processing raster {planet_raster}...")
    with rasterio.open(planet_raster) as src:
        arr = src.read(1)
        src_transform = src.transform
        src_crs = src.crs

        if src_crs is None:
            raise ValueError(f"{planet_raster} has no CRS.")

        labels = remap_labels(arr)
        labels_agg = aggregate_to_coarser_resolution(labels, scale_factor=2)
        transform_agg = aggregated_transform_2x2(src_transform)

        # Add buffering by proximity  to handle edge cases
        labels_agg_class_1 = np.where(labels_agg == 1, 1, 0).astype(np.uint8)
        labels_agg_class_0 = np.where(labels_agg == 0, 1, 0).astype(np.uint8)
        labels_agg_buffered_class_1 = buffer_by_proximity(labels_agg_class_1, buffer_size=2)
        labels_agg_buffered_class_0 = buffer_by_proximity(labels_agg_class_0, buffer_size=2)
        buffered_combined = np.where(labels_agg_buffered_class_1 == 1, 1, 255)
        buffered_combined = np.where(labels_agg_buffered_class_0 == 1, 0, buffered_combined).astype(np.uint8)

        df = sample_balanced_pixels(
            label_agg=buffered_combined,
            transform=transform_agg,
            crs=src_crs,
            frac1=frac1,
            frac0=frac0,
            max_samples_per_class=max_samples_per_class,
            seed=seed,
        )

    df["source_raster"] = str(planet_raster)

    df = add_tessera_tile_columns(df)
    df = fetch_embeddings_grouped_by_tile(
        df_points=df,
        year=tessera_year,
        cache_dir=cache_dir,
    )
    print(f"Finished processing raster {planet_raster} in {time.time() - t0:.2f} seconds.")
    return df



In [ ]:
for tile_id in tiles_id:
    output_bucket = 'ctrees-development'
    output_prefix = 'activity/tessera/logging_pantropical_v1/extracted_embeddings'
    file_name = f'embeddings_PAGE_{tile_id}.parquet'
    output_path = f's3://{output_bucket}/{output_prefix}/{file_name}'


    # check if output already exists:
    import boto3
    s3 = boto3.client('s3')
    bucket_name = output_bucket
    prefix = f'{output_prefix}/{file_name}'

    response = s3.list_objects_v2(Bucket=output_bucket, Prefix=prefix)
    if 'Contents' in response:
        print(f"Output for tile {tile_id} already exists. Skipping...")
        continue

    print(f"Processing tile {tile_id}...")
    planet_raster = planet_file_dict[tile_id]
    df = process_one_planet_raster(
        planet_raster=planet_raster,
        tessera_year=2024,
        frac1=0.5,
        frac0=0.5,
        max_samples_per_class=10000,
        seed=42,
        cache_dir=None,
    )
    print(f"Finished processing tile {tile_id}. Resulting dataframe shape: {df.shape}")

    # export to s3 as parquet:
    df.to_parquet(output_path, index=False)
    print(f"Exported processed dataframe to {output_path}")

    # remove the downloaded embeeding folder path:
    working_dir = os.getcwd()
    embedding_folder = os.path.join(working_dir, "global_0.1_degree_representation")
    landmark_folder = os.path.join(working_dir, "global_0.1_degree_tiff_all")
    if os.path.exists(embedding_folder):
        import shutil
        shutil.rmtree(embedding_folder)
        print(f"Removed embedding folder: {embedding_folder}")
    else:
        print(f"Embedding folder not found, skipping removal: {embedding_folder}")


    if os.path.exists(landmark_folder):
        import shutil
        shutil.rmtree(landmark_folder)
        print(f"Removed landmark folder: {landmark_folder}")
    else:
        print(f"Landmark folder not found, skipping removal: {landmark_folder}")

Processing tile 0691-1051...
Processing raster s3://ctrees-activity-product/logging/v1_2024_update/cumulative/2015-12-01-2025-01-01/logging_cumulative_PAGE_0691-1051_START_2015-12-01_END_2025-01-01.tif...
Remapped labels in 0.02 seconds
uniqure values after aggregation: [0.0000e+00 2.5000e-01 5.0000e-01 7.5000e-01 1.0000e+00 6.3750e+01
 6.4000e+01 6.4250e+01 6.4500e+01 1.2750e+02 1.2775e+02 1.2800e+02
 1.9125e+02 1.9150e+02 2.5500e+02]
uniqure values after final thresholding: [  0   1 255]
Updating affine transform for 2x2 aggregation...
Sampling balanced pixels with frac1=0.5, frac0=0.5, max_samples_per_class=10000...
Sampled balanced pixels in 0.08 seconds
Adding Tessera tile center columns to the dataframe...
Computing Tessera tile centers from lon/lat...
Computed Tessera tile centers in 0.00 seconds
Added Tessera tile center columns in 0.00 seconds
Fetching Tessera embeddings grouped by tile for year=2024...
Fetched Tessera embeddings and merged with points in 133.62 seconds
Finish